# Catcher D1 or Not Models — Train V2

Split the **hitter** CSV into just the catchers to do model development.

Building out the D1 or not models for catchers. Uses 8 production features including both `c_velo` and `pop_time` as defensive metrics.

**Source:** `backend/ml/train/train_v2/hitters/hitters_cleaned.csv`

In [1]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

HITTER_CSV = Path("hitters_cleaned.csv")

hitters = pd.read_csv(HITTER_CSV, low_memory=False)
print(f"Loaded {len(hitters):,} rows, {len(hitters.columns)} columns")

catchers = hitters[hitters["resolved_position"] == "C"]
print(f"Catchers: {len(catchers)}")
catchers["resolved_position"].value_counts()

Loaded 32,999 rows, 50 columns
Catchers: 6850


resolved_position
C    6850
Name: count, dtype: int64

In [2]:
C_MODELING_COLS = [
    "player_state", "resolved_position",
    "height", "weight", "throwing_hand", "hitting_handedness",
    "exit_velo_max", "exit_velo_max_date",
    "exit_velo_avg", "exit_velo_avg_date",
    "distance_max", "distance_max_date",
    "sweet_spot_p", "sweet_spot_p_date",
    "hand_speed_max", "hand_speed_max_date",
    "bat_speed_max", "bat_speed_max_date",
    "rot_acc_max", "rot_acc_max_date",
    "hard_hit_p", "hard_hit_p_date",
    "sixty_time", "sixty_time_date",
    "thirty_time", "thirty_time_date",
    "ten_yard_time", "ten_yard_time_date",
    "run_speed_max", "run_speed_max_date",
    "c_velo", "c_velo_date",
    "pop_time", "pop_time_date",
    "player_region", "committment_group", "commitment_date"
]

C_MODELING_COLS = [c for c in C_MODELING_COLS if c in catchers.columns]
c_modeling = catchers[C_MODELING_COLS]

c_modeling = c_modeling[c_modeling["committment_group"] != "Unknown"]

print(c_modeling["committment_group"].value_counts())
print(c_modeling.shape)

committment_group
Non D1       4755
Non P4 D1    1226
P4            277
Name: count, dtype: int64
(6258, 37)


In [3]:
c_modeling["d1_or_not"] = (c_modeling["committment_group"] != "Non D1").astype(int)

c_modeling["d1_or_not"].value_counts()

d1_or_not
0    4755
1    1503
Name: count, dtype: int64

In [4]:
# ============================================================
# STALE DATA ANALYSIS
# ============================================================
STALE_MONTHS = 24
OUTLIER_STD_DEV = 2

c_model_recent = c_modeling.copy()
c_model_recent["class"] = catchers.loc[c_model_recent.index, "class"]

STAT_DATE_PAIRS = [
    ("exit_velo_max",  "exit_velo_max_date"),
    ("exit_velo_avg",  "exit_velo_avg_date"),
    ("distance_max",   "distance_max_date"),
    ("sweet_spot_p",   "sweet_spot_p_date"),
    ("hand_speed_max", "hand_speed_max_date"),
    ("bat_speed_max",  "bat_speed_max_date"),
    ("rot_acc_max",    "rot_acc_max_date"),
    ("hard_hit_p",     "hard_hit_p_date"),
    ("sixty_time",     "sixty_time_date"),
    ("thirty_time",    "thirty_time_date"),
    ("ten_yard_time",  "ten_yard_time_date"),
    ("run_speed_max",  "run_speed_max_date"),
    ("c_velo",         "c_velo_date"),
    ("pop_time",       "pop_time_date"),
]
STAT_DATE_PAIRS = [(s, d) for s, d in STAT_DATE_PAIRS if s in c_model_recent.columns and d in c_model_recent.columns]

def parse_pbr_date(d):
    if pd.isna(d) or str(d).strip() == "":
        return pd.NaT
    try:
        return pd.to_datetime(str(d).strip(), format="%m/%d/%y")
    except Exception:
        try:
            return pd.to_datetime(str(d).strip())
        except Exception:
            return pd.NaT

c_model_recent["grad_date"] = pd.to_datetime(c_model_recent["class"].astype(str) + "-06-01")

for stat_col, date_col in STAT_DATE_PAIRS:
    parsed_col = f"_{date_col}_parsed"
    months_col = f"{stat_col}__months_before_grad"
    c_model_recent[parsed_col] = c_model_recent[date_col].apply(parse_pbr_date)
    c_model_recent[months_col] = ((c_model_recent["grad_date"] - c_model_recent[parsed_col]).dt.days / 30.44).round(1)

print(f"Parsed {len(STAT_DATE_PAIRS)} stat date columns.\n")

print(f"{'='*70}")
print(f"STALENESS BY COMMITMENT GROUP  (threshold = {STALE_MONTHS} months)")
print(f"{'='*70}\n")

for group in ["P4", "Non P4 D1", "Non D1"]:
    mask = c_model_recent["committment_group"] == group
    g_stale = 0
    g_total = 0
    for stat_col, _ in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        has_val = c_model_recent.loc[mask, stat_col].notna()
        is_stale = c_model_recent.loc[mask, months_col] > STALE_MONTHS
        g_stale += (has_val & is_stale).sum()
        g_total += has_val.sum()
    pct = 100 * g_stale / g_total if g_total else 0
    n_players = mask.sum()
    print(f"  {group:>12}  ({n_players:>5,} players): {g_stale:>4,} / {g_total:>5,} values stale ({pct:.1f}%)")

stat_value_cols = [s for s, _ in STAT_DATE_PAIRS]
print(f"\n{'='*70}")
print(f"OUTLIER SUMMARY  (+-{OUTLIER_STD_DEV} std dev from group mean)")
print(f"{'='*70}\n")

for group in ["P4", "Non P4 D1", "Non D1"]:
    mask = c_model_recent["committment_group"] == group
    tot_outliers = 0
    tot_stale_o = 0
    for stat_col in stat_value_cols:
        months_col = f"{stat_col}__months_before_grad"
        vals = c_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale = c_model_recent.loc[mask, months_col] > STALE_MONTHS
        tot_outliers += is_outlier.sum()
        tot_stale_o += (is_outlier & is_stale).sum()
    tot_fresh_o = tot_outliers - tot_stale_o
    pct = 100 * tot_stale_o / tot_outliers if tot_outliers else 0
    print(f"  {group:>12}: {tot_outliers:>4} outliers -> {tot_stale_o:>3} stale ({pct:.0f}%), {tot_fresh_o:>3} fresh ({100-pct:.0f}%)")

internal_cols = [c for c in c_model_recent.columns if c.startswith("_")]
c_model_recent.drop(columns=internal_cols, inplace=True)
print(f"\nc_model_recent shape: {c_model_recent.shape}")

Parsed 14 stat date columns.

STALENESS BY COMMITMENT GROUP  (threshold = 24 months)

            P4  (  277 players):  792 / 2,699 values stale (29.3%)
     Non P4 D1  (1,226 players): 3,051 / 11,680 values stale (26.1%)
        Non D1  (4,755 players): 8,175 / 45,173 values stale (18.1%)

OUTLIER SUMMARY  (+-2 std dev from group mean)

            P4:  118 outliers ->  74 stale (63%),  44 fresh (37%)
     Non P4 D1:  503 outliers -> 246 stale (49%), 257 fresh (51%)
        Non D1: 2023 outliers -> 652 stale (32%), 1371 fresh (68%)

c_model_recent shape: (6258, 54)


In [5]:
# ============================================================
# APPLY OPTION B — NaN only stale outliers (MULTI-PASS)
# ============================================================
MAX_PASSES = 5
grand_total = 0

for pass_i in range(1, MAX_PASSES + 1):
    pass_total = 0
    pass_log = []

    for stat_col, date_col in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        stat_nand = 0

        for group in ["P4", "Non P4 D1", "Non D1"]:
            mask = c_model_recent["committment_group"] == group
            vals = c_model_recent.loc[mask, stat_col]
            mean = vals.mean()
            std = vals.std()

            if pd.isna(mean) or pd.isna(std) or std == 0:
                continue

            is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
            is_stale = c_model_recent.loc[mask, months_col] > STALE_MONTHS

            to_nan = is_outlier & is_stale
            n = to_nan.sum()

            if n > 0:
                c_model_recent.loc[to_nan[to_nan].index, stat_col] = np.nan
                c_model_recent.loc[to_nan[to_nan].index, date_col] = np.nan
                stat_nand += n

        if stat_nand > 0:
            pass_log.append({"stat": stat_col, "values_removed": stat_nand})
            pass_total += stat_nand

    grand_total += pass_total
    print(f"Pass {pass_i}: NaN'd {pass_total} stale outlier values")
    if pass_log:
        for row in pass_log:
            print(f"    {row['stat']:>15}: {row['values_removed']}")

    if pass_total == 0:
        print(f"  No more stale outliers found — stopping.")
        break

print(f"\nTotal across {pass_i} passes: {grand_total} values NaN'd")

remaining = 0
for stat_col, _ in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    for group in ["P4", "Non P4 D1", "Non D1"]:
        mask = c_model_recent["committment_group"] == group
        vals = c_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale = c_model_recent.loc[mask, months_col] > STALE_MONTHS
        remaining += (is_outlier & is_stale).sum()

print(f"Stale outliers remaining: {remaining}")
print(f"c_model_recent shape: {c_model_recent.shape}")

Pass 1: NaN'd 972 stale outlier values
      exit_velo_max: 139
      exit_velo_avg: 76
       distance_max: 83
       sweet_spot_p: 88
     hand_speed_max: 55
      bat_speed_max: 75
        rot_acc_max: 50
         hard_hit_p: 8
         sixty_time: 94
        thirty_time: 19
      ten_yard_time: 15
      run_speed_max: 27
             c_velo: 116
           pop_time: 127
Pass 2: NaN'd 252 stale outlier values
      exit_velo_max: 52
      exit_velo_avg: 28
       distance_max: 26
       sweet_spot_p: 4
     hand_speed_max: 9
      bat_speed_max: 17
        rot_acc_max: 8
         hard_hit_p: 3
         sixty_time: 26
        thirty_time: 5
      ten_yard_time: 4
      run_speed_max: 5
             c_velo: 28
           pop_time: 37
Pass 3: NaN'd 47 stale outlier values
      exit_velo_max: 19
      exit_velo_avg: 5
       distance_max: 5
       sweet_spot_p: 1
     hand_speed_max: 1
      bat_speed_max: 3
         sixty_time: 8
        thirty_time: 1
           pop_time: 4
Pass 4: N

## Logistic Regression Baseline

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

FEATURES = [
    "height", "weight",
    "exit_velo_max", "distance_max", "bat_speed_max",
    "sixty_time",
    "c_velo", "pop_time",
]

X = c_model_recent[FEATURES]
y = c_model_recent["d1_or_not"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("impute", KNNImputer(n_neighbors=10)),
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred), "\n")

coefs = pd.Series(pipe["lr"].coef_[0], index=FEATURES).sort_values()
print(coefs)

              precision    recall  f1-score   support

      Non D1       0.89      0.69      0.78       951
          D1       0.43      0.73      0.54       301

    accuracy                           0.70      1252
   macro avg       0.66      0.71      0.66      1252
weighted avg       0.78      0.70      0.72      1252

[[660 291]
 [ 80 221]] 

pop_time        -0.399941
sixty_time      -0.232542
exit_velo_max   -0.011695
bat_speed_max    0.038628
weight           0.045544
height           0.226834
distance_max     0.380380
c_velo           0.454519
dtype: float64


## XGBoost — 8 Production Features + Calibration

In [12]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

FEATURES_XGB = [
    "height", "weight",
    "exit_velo_max", "distance_max",
    "sixty_time",
    "c_velo", "pop_time",
    "bat_speed_max",
]

X = c_model_recent[FEATURES_XGB]
y = c_model_recent["d1_or_not"]

neg, pos = (y == 0).sum(), (y == 1).sum()
print(f"Class balance: {neg} Non D1, {pos} D1  (ratio {neg/pos:.2f}:1)\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=3.0,
    min_child_weight=5,
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42,
    enable_categorical=False,
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]

print("XGBoost — 8 Features (holdout)")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred), "\n")

auc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC: {auc:.4f}")
print(f"  -> {auc:.0%} of the time, a random D1 player scores higher than a random Non-D1 player\n")

importances = pd.Series(xgb.feature_importances_, index=FEATURES_XGB).sort_values(ascending=False)
print(importances)

# Stratified 5-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    xgb, X, y, cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall", "accuracy"],
    return_train_score=True,
)

print(f"\n\nStratified 5-Fold CV:")
print("=" * 55)
for metric in ["roc_auc", "accuracy", "f1", "precision", "recall"]:
    train_scores = cv_results[f"train_{metric}"]
    test_scores = cv_results[f"test_{metric}"]
    label = "AUC-ROC" if metric == "roc_auc" else metric
    print(f"  {label:>12}:  train {train_scores.mean():.3f} (+/- {train_scores.std():.3f})  |  test {test_scores.mean():.3f} (+/- {test_scores.std():.3f})")

print(f"\n  Overfit gap (AUC): {cv_results['train_roc_auc'].mean() - cv_results['test_roc_auc'].mean():.3f}")

# Compare with LogReg baseline
xgb_report = classification_report(y_test, y_pred, target_names=["Non D1", "D1"], output_dict=True)
print(f"\n{'=' * 55}")
print("COMPARISON: LogReg (8 feat) vs XGBoost (8 feat)")
print(f"{'=' * 55}")
print(f"  XGBoost 8-feat:   D1 precision={xgb_report['D1']['precision']:.2f}  "
      f"D1 recall={xgb_report['D1']['recall']:.2f}  "
      f"D1 F1={xgb_report['D1']['f1-score']:.2f}  "
      f"accuracy={xgb_report['accuracy']:.2f}  "
      f"AUC={auc:.3f}")

# ============================================================
# Calibrated XGBoost — isotonic calibration
# ============================================================
xgb_calibrated = CalibratedClassifierCV(xgb, method="isotonic", cv=5)
xgb_calibrated.fit(X_train, y_train)

y_proba_cal = xgb_calibrated.predict_proba(X_test)[:, 1]
y_pred_cal = (y_proba_cal >= 0.45).astype(int)

print("\n\nCalibrated XGBoost — 8 Features (threshold=0.45)")
print("=" * 60)
print(classification_report(y_test, y_pred_cal, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred_cal), "\n")

fraction_raw, mean_raw = calibration_curve(y_test, y_proba, n_bins=10)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=10)

print("Calibration comparison (raw vs calibrated):")
print("=" * 75)
print(f"  {'--- RAW ---':^28}   |   {'--- CALIBRATED ---':^28}")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}   |   {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
print(f"  {'-'*28}   |   {'-'*28}")

max_rows = max(len(mean_raw), len(mean_cal))
for i in range(max_rows):
    if i < len(mean_raw):
        r_gap = mean_raw[i] - fraction_raw[i]
        raw_str = f"  {mean_raw[i]:>10.2f}  {fraction_raw[i]:>8.2f}  {r_gap:>+8.2f}"
    else:
        raw_str = f"  {'':>28}"
    if i < len(mean_cal):
        c_gap = mean_cal[i] - fraction_cal[i]
        cal_str = f"{mean_cal[i]:>10.2f}  {fraction_cal[i]:>8.2f}  {c_gap:>+8.2f}"
        flag = "  <- off" if abs(c_gap) > 0.10 else ""
        cal_str += flag
    else:
        cal_str = f"{'':>28}"
    print(f"{raw_str}   |   {cal_str}")

brier_raw = brier_score_loss(y_test, y_proba)
brier_cal = brier_score_loss(y_test, y_proba_cal)
print(f"\nBrier score:  raw = {brier_raw:.4f}  |  calibrated = {brier_cal:.4f}  |  improvement = {brier_raw - brier_cal:.4f}")

max_gap_raw = max(abs(mean_raw[i] - fraction_raw[i]) for i in range(len(mean_raw)))
max_gap_cal = max(abs(mean_cal[i] - fraction_cal[i]) for i in range(len(mean_cal)))
print(f"Max calibration gap:  raw = {max_gap_raw:.2f}  |  calibrated = {max_gap_cal:.2f}")
print(f"Worst-case Non-D1 PCI error:  raw = {max_gap_raw * 25:.1f} pts  |  calibrated = {max_gap_cal * 25:.1f} pts")

auc_cal = roc_auc_score(y_test, y_proba_cal)
print(f"AUC-ROC (calibrated): {auc_cal:.4f}")

Class balance: 4755 Non D1, 1503 D1  (ratio 3.16:1)

XGBoost — 8 Features (holdout)
              precision    recall  f1-score   support

      Non D1       0.92      0.69      0.79       951
          D1       0.45      0.80      0.58       301

    accuracy                           0.72      1252
   macro avg       0.68      0.75      0.68      1252
weighted avg       0.80      0.72      0.74      1252

[[660 291]
 [ 61 240]] 

AUC-ROC: 0.8215
  -> 82% of the time, a random D1 player scores higher than a random Non-D1 player

c_velo           0.298659
pop_time         0.177365
distance_max     0.136513
exit_velo_max    0.124745
height           0.078813
sixty_time       0.073169
weight           0.064009
bat_speed_max    0.046726
dtype: float32


Stratified 5-Fold CV:
       AUC-ROC:  train 0.841 (+/- 0.002)  |  test 0.795 (+/- 0.007)
      accuracy:  train 0.726 (+/- 0.004)  |  test 0.699 (+/- 0.012)
            f1:  train 0.588 (+/- 0.002)  |  test 0.548 (+/- 0.008)
     precisio

## Feature Engineering — Interaction Features

In [8]:
CORE_8 = ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "c_velo", "pop_time", "bat_speed_max"]

c_eng = c_model_recent[CORE_8 + ["d1_or_not"]].copy()

# --- Interaction features ---
# Catcher defense: arm strength + quick release (pop_time is lower=better)
c_eng["arm_pop_ratio"] = c_eng["c_velo"] / c_eng["pop_time"]

# Raw two-way tools: arm strength x bat power
c_eng["velo_x_exit"] = c_eng["c_velo"] * c_eng["exit_velo_max"]

# Power tools: bat speed x exit velo = swing-to-contact transfer
c_eng["bat_x_exit"] = c_eng["bat_speed_max"] * c_eng["exit_velo_max"]

# Distance x arm velo: overall power + arm combo
c_eng["dist_x_arm"] = c_eng["distance_max"] * c_eng["c_velo"]

# Body mass index proxy
c_eng["bmi_proxy"] = c_eng["weight"] / (c_eng["height"] ** 2) * 703

# Speed-adjusted power: exit velo relative to sixty time
c_eng["exit_per_sixty"] = c_eng["exit_velo_max"] / c_eng["sixty_time"]

# --- Check separation ---
ENG_FEATURES = [c for c in c_eng.columns if c not in ["d1_or_not"]]

print(f"c_eng: {c_eng.shape[0]} rows, {len(ENG_FEATURES)} features")
print(f"  Core: {len(CORE_8)}  |  Engineered: {len(ENG_FEATURES) - len(CORE_8)}\n")

print("Cohen's d (sorted by |d|):")
print("=" * 55)
effects = []
for feat in ENG_FEATURES:
    d1 = c_eng.loc[c_eng["d1_or_not"] == 1, feat].dropna()
    nd1 = c_eng.loc[c_eng["d1_or_not"] == 0, feat].dropna()
    pooled = np.sqrt((nd1.std()**2 + d1.std()**2) / 2)
    d = (d1.mean() - nd1.mean()) / pooled if pooled > 0 else 0
    effects.append((feat, d))
for feat, d in sorted(effects, key=lambda x: abs(x[1]), reverse=True):
    tag = " *" if feat not in CORE_8 else ""
    print(f"  {feat:>20}  d = {d:+.3f}{tag}")

print("\n* = engineered feature")

print("\nEngineered feature correlations with parents:")
print("=" * 55)
eng_pairs = [
    ("arm_pop_ratio", ["c_velo", "pop_time"]),
    ("velo_x_exit",   ["c_velo", "exit_velo_max"]),
    ("bat_x_exit",    ["bat_speed_max", "exit_velo_max"]),
    ("dist_x_arm",    ["distance_max", "c_velo"]),
    ("exit_per_sixty", ["exit_velo_max", "sixty_time"]),
    ("bmi_proxy",     ["height", "weight"]),
]
for eng, parents in eng_pairs:
    corrs = [c_eng[eng].corr(c_eng[p]) for p in parents]
    corr_str = ", ".join(f"{p}={c:.2f}" for p, c in zip(parents, corrs))
    print(f"  {eng:>20} -> {corr_str}")

c_eng: 6258 rows, 14 features
  Core: 8  |  Engineered: 6

Cohen's d (sorted by |d|):
            dist_x_arm  d = +1.140 *
           velo_x_exit  d = +1.130 *
         arm_pop_ratio  d = +1.120 *
                c_velo  d = +1.037
        exit_per_sixty  d = +0.958 *
              pop_time  d = -0.945
          distance_max  d = +0.925
         exit_velo_max  d = +0.890
            bat_x_exit  d = +0.813 *
            sixty_time  d = -0.665
         bat_speed_max  d = +0.612
                weight  d = +0.438
                height  d = +0.433
             bmi_proxy  d = +0.197 *

* = engineered feature

Engineered feature correlations with parents:
         arm_pop_ratio -> c_velo=0.91, pop_time=-0.91
           velo_x_exit -> c_velo=0.87, exit_velo_max=0.90
            bat_x_exit -> bat_speed_max=0.93, exit_velo_max=0.88
            dist_x_arm -> distance_max=0.95, c_velo=0.75
        exit_per_sixty -> exit_velo_max=0.87, sixty_time=-0.78
             bmi_proxy -> height=-0.09, weig

In [9]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

# ============================================================
# XGBoost — Engineered Features + Calibration
# ============================================================
FEATURES_ENG = [c for c in c_eng.columns if c != "d1_or_not"]

X = c_eng[FEATURES_ENG]
y = c_eng["d1_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_eng = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=3.0,
    min_child_weight=5,
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42,
    enable_categorical=False,
)

xgb_eng.fit(X_train, y_train)
y_pred = xgb_eng.predict(X_test)
y_proba = xgb_eng.predict_proba(X_test)[:, 1]

print(f"XGBoost — {len(FEATURES_ENG)} Features (8 core + {len(FEATURES_ENG)-8} engineered)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred))

auc_eng = roc_auc_score(y_test, y_proba)
print(f"\nAUC-ROC: {auc_eng:.4f}")

importances = pd.Series(xgb_eng.feature_importances_, index=FEATURES_ENG).sort_values(ascending=False)
print(f"\nFeature importance:")
print(importances.to_string())

# Stratified 5-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_validate(
    xgb_eng, X, y, cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall", "accuracy"],
    return_train_score=True,
)

print(f"\nStratified 5-Fold CV:")
print("=" * 60)
for metric in ["roc_auc", "accuracy", "f1", "precision", "recall"]:
    tr = cv[f"train_{metric}"]
    te = cv[f"test_{metric}"]
    label = "AUC-ROC" if metric == "roc_auc" else metric
    print(f"  {label:>12}:  train {tr.mean():.3f} (+/- {tr.std():.3f})  |  test {te.mean():.3f} (+/- {te.std():.3f})")
print(f"\n  Overfit gap (AUC): {cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean():.3f}")

# Calibration
xgb_eng_cal = CalibratedClassifierCV(xgb_eng, method="isotonic", cv=5)
xgb_eng_cal.fit(X_train, y_train)
y_proba_cal = xgb_eng_cal.predict_proba(X_test)[:, 1]
y_pred_cal = (y_proba_cal >= 0.45).astype(int)

fraction_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=10)
brier_raw = brier_score_loss(y_test, y_proba)
brier_cal = brier_score_loss(y_test, y_proba_cal)

print(f"\nCalibrated model (threshold=0.45):")
print("=" * 60)
print(classification_report(y_test, y_pred_cal, target_names=["Non D1", "D1"]))
print(f"Brier:  raw = {brier_raw:.4f}  |  calibrated = {brier_cal:.4f}")
print(f"\nCalibration curve (calibrated):")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
for pred, actual in zip(mean_cal, fraction_cal):
    gap = pred - actual
    flag = "  <- off" if abs(gap) > 0.10 else ""
    print(f"  {pred:>10.2f}  {actual:>8.2f}  {gap:>+8.2f}{flag}")

auc_eng_cal = roc_auc_score(y_test, y_proba_cal)
print(f"\nAUC-ROC (calibrated): {auc_eng_cal:.4f}")

print(f"\n{'=' * 60}")
print(f"COMPARISON: 8-feature base vs {len(FEATURES_ENG)}-feature engineered")
print(f"{'=' * 60}")
print(f"  8-feat base:      AUC={auc:.4f}  Brier(cal)={brier_cal:.4f}")
print(f"  {len(FEATURES_ENG)}-feat eng:     AUC={auc_eng:.4f}  Brier(cal)={brier_cal:.4f}")
print(f"  AUC delta:        {auc_eng - auc:+.4f}")

XGBoost — 14 Features (8 core + 6 engineered)
              precision    recall  f1-score   support

      Non D1       0.92      0.70      0.79       951
          D1       0.46      0.80      0.58       301

    accuracy                           0.72      1252
   macro avg       0.69      0.75      0.69      1252
weighted avg       0.81      0.72      0.74      1252

[[667 284]
 [ 61 240]]

AUC-ROC: 0.8213

Feature importance:
velo_x_exit       0.208975
arm_pop_ratio     0.163215
dist_x_arm        0.122483
c_velo            0.083823
pop_time          0.059301
exit_per_sixty    0.051270
distance_max      0.048816
height            0.047388
exit_velo_max     0.042634
weight            0.041523
sixty_time        0.039569
bat_x_exit        0.034330
bmi_proxy         0.030333
bat_speed_max     0.026340

Stratified 5-Fold CV:
       AUC-ROC:  train 0.849 (+/- 0.002)  |  test 0.796 (+/- 0.009)
      accuracy:  train 0.737 (+/- 0.003)  |  test 0.710 (+/- 0.008)
            f1:  train 0.600 

In [ ]:
import joblib
import json
import os
from datetime import datetime

# ============================================================
# Save calibrated XGBoost D1 model to production model directory
# ============================================================
THRESHOLD = 0.45
VERSION = f"version_{datetime.now().strftime('%m%d%Y')}"
SAVE_DIR = os.path.abspath(os.path.join(
    os.getcwd(), "..", "..", "..", "models", "models_c", "models_d1_or_not_c", VERSION
))
os.makedirs(SAVE_DIR, exist_ok=True)

FEATURES_XGB = ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "c_velo", "pop_time", "bat_speed_max"]

X = c_model_recent[FEATURES_XGB]
y = c_model_recent["d1_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_final = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=3.0,
    min_child_weight=5,
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42,
    enable_categorical=False,
)
xgb_final.fit(X_train, y_train)

xgb_final_cal = CalibratedClassifierCV(xgb_final, method="isotonic", cv=5)
xgb_final_cal.fit(X_train, y_train)

# --- Eval metrics on holdout ---
y_proba_test = xgb_final_cal.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= THRESHOLD).astype(int)

auc = roc_auc_score(y_test, y_proba_test)
brier = brier_score_loss(y_test, y_proba_test)
report = classification_report(y_test, y_pred_test, target_names=["Non D1", "D1"], output_dict=True)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_test, n_bins=10)
max_cal_gap = max(abs(m - f) for m, f in zip(mean_cal, fraction_cal))

# --- Save model ---
joblib.dump(xgb_final_cal, os.path.join(SAVE_DIR, "calibrated_xgb_model.pkl"))
print(f"Saved: calibrated_xgb_model.pkl")

# --- Save model config ---
model_config = {
    "model_version": f"c_d1_xgb_cal_{VERSION}",
    "creation_date": datetime.now().isoformat(),
    "model_type": "calibrated_xgboost_catcher_d1",
    "calibration_method": "isotonic",
    "threshold": THRESHOLD,
    "features": FEATURES_XGB,
    "hyperparameters": {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.01,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 1.0,
        "reg_lambda": 3.0,
        "min_child_weight": 5,
        "scale_pos_weight": round(neg / pos, 4),
    },
    "performance_metrics": {
        "auc_roc": round(auc, 4),
        "brier_score": round(brier, 4),
        "max_calibration_gap": round(max_cal_gap, 4),
        "d1_precision": round(report["D1"]["precision"], 4),
        "d1_recall": round(report["D1"]["recall"], 4),
        "d1_f1": round(report["D1"]["f1-score"], 4),
        "accuracy": round(report["accuracy"], 4),
        "threshold_used": THRESHOLD,
    },
    "dataset_info": {
        "total_samples": len(c_model_recent),
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "d1_rate": round(pos / (neg + pos), 4),
        "stale_cleaning": "Option B multi-pass (stale outliers only, +/-2 std + >24mo)",
    },
    "calibration_curve": {
        "predicted": [round(float(m), 4) for m in mean_cal],
        "actual": [round(float(f), 4) for f in fraction_cal],
    },
}

with open(os.path.join(SAVE_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)
print(f"Saved: model_config.json")

# --- Save feature metadata ---
feature_metadata = {
    "features": FEATURES_XGB,
    "required_columns": FEATURES_XGB,
    "notes": "XGBoost handles NaN natively. No imputation or scaling needed. 8 features including both c_velo and pop_time as defensive metrics.",
}

with open(os.path.join(SAVE_DIR, "feature_metadata.json"), "w") as f:
    json.dump(feature_metadata, f, indent=2)
print(f"Saved: feature_metadata.json")

# --- Print summary ---
print(f"\nModel saved to: {SAVE_DIR}")
print(f"\n{'=' * 60}")
print(f"PRODUCTION MODEL METRICS (threshold={THRESHOLD})")
print(f"{'=' * 60}")
print(f"  AUC-ROC:           {auc:.4f}")
print(f"  Brier score:       {brier:.4f}")
print(f"  Max cal gap:       {max_cal_gap:.4f}")
print(f"  D1 Precision:      {report['D1']['precision']:.4f}")
print(f"  D1 Recall:         {report['D1']['recall']:.4f}")
print(f"  D1 F1:             {report['D1']['f1-score']:.4f}")
print(f"  Accuracy:          {report['accuracy']:.4f}")
print(f"  Features:          {len(FEATURES_XGB)} ({', '.join(FEATURES_XGB)})")
print(f"  Threshold:         {THRESHOLD}")
print(f"  Training samples:  {len(X_train)}")
print(f"  Test samples:      {len(X_test)}")
print(f"  D1 base rate:      {pos/(neg+pos):.1%}")

print(f"\nCalibration curve:")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
for pred, actual in zip(mean_cal, fraction_cal):
    gap = pred - actual
    flag = "  <- off" if abs(gap) > 0.10 else ""
    print(f"  {pred:>10.2f}  {actual:>8.2f}  {gap:>+8.2f}{flag}")

print(f"\nFiles in {VERSION}/:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:<35} {size:>8,} bytes")